In [ ]:
%pip install requests
%pip install pandas

In [85]:
import os
import typing
import datetime

import requests
import pandas as pd

In [2]:
DATA_DIR:str            = "./data"

DATA_DIR_RAW:str        = DATA_DIR + "/raw"
DATA_DIR_MODIFIED:str   = DATA_DIR + "/modified"
DATA_DIR_TAG:str        = DATA_DIR + "/tag"

In [3]:
if not os.path.isdir(DATA_DIR):
    os.makedirs(DATA_DIR)

if not os.path.isdir(DATA_DIR_RAW):
    os.makedirs(DATA_DIR_RAW)

if not os.path.isdir(DATA_DIR_MODIFIED):
    os.makedirs(DATA_DIR_MODIFIED)

if not os.path.isdir(DATA_DIR_TAG):
    os.makedirs(DATA_DIR_TAG)

In [4]:
REQUEST_URL:str = "https://solved.ac/api/v3/problem/lookup"
REQUEST_PARAMS_KEY:list = ["problemIds"]

In [10]:
date_str = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

In [98]:
def CrawlingandSaveRawData(num_start:int, num_end:int) -> tuple[pd.DataFrame, str]:
    def getProblemIdsParam(num_from:int, num_to:int) -> str:
        result:str = ""
        for _ in range(num_from, num_to, 1):
            result += f"{_},"
        return result[:-1]

    response:requests.Response = requests.get(url=REQUEST_URL, params=dict(zip(REQUEST_PARAMS_KEY, [getProblemIdsParam(num_start, num_end)])))
    if response.status_code == 400:
        raise requests.RequestException('Request 400')
    
    responseBody = response.json()
    responseBodyDict:dict = {}
    for _ in responseBody:
        responseBodyDict[_["problemId"]] = _

    df_columns:list = list(responseBodyDict[list(responseBodyDict.keys())[0]].keys())

    df_data:list = []
    for _ in responseBodyDict.keys():
        df_data_row:list = []
        for data in df_columns:
            df_data_row.append(responseBodyDict[_][data])
        df_data.append(df_data_row)

    df = pd.DataFrame(df_data, columns=df_columns)
    df.index = df["problemId"]
    del df["problemId"]

    file_raw_name:str = f"{DATA_DIR_RAW}/boj.problem.raw.{num_start}.{num_end-1}.{date_str}.csv"
    df.to_csv(file_raw_name)

    return (df, file_raw_name)

In [105]:
def ModifyAndSaveData(num_start:int, num_end:int, df:pd.DataFrame) -> str:
    title_en:list[str] = []
    for i in list(df.index):
        temp = ""
        for title in df.loc[i]["titles"]:
            if title["language"] == "en":
                temp = title["title"]
                break
        title_en.append(temp)

    tags:list[list[dict]] = []
    for i in list(df.index):
        tag:list[dict] = []
        for tagline in df.loc[i]["tags"]:
            tag.append({
                "tagBOJ": tagline["bojTagId"],
                "tagSA" : tagline["key"]
            })
        tags.append(tag)

    df = df.rename(columns={"titles": "titleEn"})
    del df["isSolvable"]
    del df["votedUserCount"]
    del df["givesNoRating"]
    del df["isLevelLocked"]
    del df["official"]
    del df["metadata"]
    del df["isPartial"]
    
    df["titleEn"] = title_en
    df["tags"] = tags

    file_modified_name:str = f"{DATA_DIR_MODIFIED}/boj.problem.modified.{num_start}.{num_end-1}.{date_str}.csv"
    df.to_csv(file_modified_name)

    return file_modified_name

In [108]:
def ExtractAndSaveTag(num_start:int, num_end:int, df:pd.DataFrame) -> str:
    tags_modified:list[list] = []

    tags_raw = df["tags"].copy()
    for id in list(tags_raw.keys()):
        for tag in tags_raw.loc[id]:
            tag_modified:list = [tag["bojTagId"], tag["key"]]
            tag_name_ko:str = ""
            tag_name_en:str = ""
            for dname in tag["displayNames"]:
                if dname["language"] == "ko":
                    tag_name_ko = dname["name"]
                if dname["language"] == "en":
                    tag_name_en = dname["name"]
            tag_modified.append(tag_name_ko)
            tag_modified.append(tag_name_en)
            tag_modified.append(tag["aliases"])
            if not tag_modified in tags_modified:
                tags_modified.append(tag_modified)
    tags_modified = sorted(tags_modified, key=lambda _ : _[0])

    df_tags = pd.DataFrame(tags_modified, columns=["tagBOJ", "tagSA", "nameKo", "nameEn", "aliases"])
    df_tags.index = df_tags["tagBOJ"]
    del df_tags["tagBOJ"]

    file_tag_name:str = f"{DATA_DIR_TAG}/boj.problem.tags.{num_start}.{num_end-1}.{date_str}.csv"
    df_tags.to_csv(file_tag_name)

    return file_tag_name

In [109]:
print("Crawling and Save Raw Data as Files")
for num_start in range(1000, 32001, 100):
    df, file_raw_name = CrawlingandSaveRawData(num_start=num_start, num_end=num_start+100)
    print(f"{num_start}~{num_start+99} -> RAW: {file_raw_name}")
    file_modified_name = ModifyAndSaveData(num_start=num_start, num_end=num_start+100, df=df.copy())
    print(f"{num_start}~{num_start+99} -> Modified: {file_modified_name}")
    file_tag_name = ExtractAndSaveTag(num_start=num_start, num_end=num_start+100, df=df)
    print(f"{num_start}~{num_start+99} -> Tag: {file_tag_name}")

Crawling and Save Raw Data as Files
1000~1099 -> RAW: ./data/raw/boj.problem.raw.1000.1099.20240624202848.csv
1000~1099 -> Modified: ./data/modified/boj.problem.modified.1000.1099.20240624202848.csv
1000~1099 -> Tag: ./data/tag/boj.problem.tags.1000.1099.20240624202848.csv
1100~1199 -> RAW: ./data/raw/boj.problem.raw.1100.1199.20240624202848.csv
1100~1199 -> Modified: ./data/modified/boj.problem.modified.1100.1199.20240624202848.csv
1100~1199 -> Tag: ./data/tag/boj.problem.tags.1100.1199.20240624202848.csv
1200~1299 -> RAW: ./data/raw/boj.problem.raw.1200.1299.20240624202848.csv
1200~1299 -> Modified: ./data/modified/boj.problem.modified.1200.1299.20240624202848.csv
1200~1299 -> Tag: ./data/tag/boj.problem.tags.1200.1299.20240624202848.csv
1300~1399 -> RAW: ./data/raw/boj.problem.raw.1300.1399.20240624202848.csv
1300~1399 -> Modified: ./data/modified/boj.problem.modified.1300.1399.20240624202848.csv
1300~1399 -> Tag: ./data/tag/boj.problem.tags.1300.1399.20240624202848.csv
1400~1499 ->

In [134]:
file_raw_list = sorted([_ for _ in list(os.listdir(DATA_DIR_RAW)) if _[-4:] == ".csv"], key=lambda x : (len(x), x))

df:pd.DataFrame = pd.read_csv(os.path.join(DATA_DIR_RAW, file_raw_list[0]), index_col="problemId")
for file in file_raw_list[1:]:
    df = pd.concat([df, pd.read_csv(os.path.join(DATA_DIR_RAW, file), index_col="problemId")], axis=0)

file_raw_name:str = f"{DATA_DIR}/boj.problem.raw.csv"
df.to_csv(file_raw_name)

In [136]:
file_modified_list = sorted([_ for _ in list(os.listdir(DATA_DIR_MODIFIED)) if _[-4:] == ".csv"], key=lambda x : (len(x), x))

df:pd.DataFrame = pd.read_csv(os.path.join(DATA_DIR_MODIFIED, file_modified_list[0]), index_col="problemId")
for file in file_modified_list[1:]:
    df = pd.concat([df, pd.read_csv(os.path.join(DATA_DIR_MODIFIED, file), index_col="problemId")], axis=0)

file_modified_name:str = f"{DATA_DIR}/boj.problem.modified.csv"
df.to_csv(file_modified_name)

In [142]:
file_tag_list = sorted([_ for _ in list(os.listdir(DATA_DIR_TAG)) if _[-4:] == ".csv"], key=lambda x : (len(x), x))

df:pd.DataFrame = pd.read_csv(os.path.join(DATA_DIR_TAG, file_tag_list[0]), index_col="tagBOJ")
for file in file_tag_list[1:]:
    df = pd.concat([df, pd.read_csv(os.path.join(DATA_DIR_TAG, file), index_col="tagBOJ")], axis=0)

df = df.sort_index()
df = df.drop_duplicates()

file_tag_name:str = f"{DATA_DIR}/boj.problem.tag.csv"
df.to_csv(file_tag_name)